In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)

IMG_ROOT = '../data/Images'
IMG_SIZE = (128, 128)
label_map = {
    'Negative': 'negative',
    'Neutral':  'neutral',
    'positive': 'positive',
}

## Load all images into memory

In [ ]:
def load_image(path):
    img = Image.open(path).convert('RGB')
    img = img.resize(IMG_SIZE, Image.LANCZOS)
    return np.array(img, dtype='float32') / 255.0

records = []
for folder, label in label_map.items():
    folder_path = os.path.join(IMG_ROOT, folder)
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            records.append({'path': os.path.join(folder_path, fname), 'label': label})

df = pd.DataFrame(records)

print(f'Loading {len(df)} images...')
X = np.array([load_image(p) for p in df['path']])  # shape: (N, 128, 128, 3)
print(f'Loaded. Array shape: {X.shape}  Memory: {X.nbytes / 1e9:.2f} GB')

le = LabelEncoder()
y = le.fit_transform(df['label'])
print('Classes:', le.classes_)

## Train/val/test split

In [ ]:
# Test set locked in — identical to all other image notebooks
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Val set carved from training portion only
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.2,
    random_state=42,
    stratify=y_trainval
)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

## Tune base filters on validation set

Architecture per candidate:
- Conv(F, 3×3) → BatchNorm → MaxPool
- Conv(2F, 3×3) → BatchNorm → MaxPool
- Conv(4F, 3×3) → BatchNorm → MaxPool
- Flatten → Dense(256) → Dropout(0.5) → Dense(3, softmax)

In [ ]:
def build_cnn(base_filters):
    return Sequential([
        Conv2D(base_filters, (3, 3), activation='relu', padding='same', input_shape=(128, 128, 3)),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Conv2D(base_filters * 2, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Conv2D(base_filters * 4, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(3, activation='softmax')
    ])

t0 = time.time()
filter_options = [32, 64]
val_results = []
es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

for f in filter_options:
    model = build_cnn(f)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=20,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )
    val_acc = model.evaluate(X_val, y_val, verbose=0)[1]
    val_results.append({'base_filters': f, 'val_acc': val_acc})
    print(f'base_filters={f:<4}  Val Accuracy: {val_acc:.3f}')

best_filters = max(val_results, key=lambda x: x['val_acc'])['base_filters']
print(f'\nBest base_filters: {best_filters}')

## Retrain on full train+val, evaluate on test

In [ ]:
best_model = build_cnn(best_filters)
best_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
best_model.fit(
    X_trainval, y_trainval,
    epochs=100,
    batch_size=32,
    callbacks=[EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)],
    verbose=1
)
runtime = time.time() - t0

y_pred_enc = np.argmax(best_model.predict(X_test), axis=1)
y_pred = le.inverse_transform(y_pred_enc)
y_test_labels = le.inverse_transform(y_test)

print(f'\nAccuracy: {accuracy_score(y_test_labels, y_pred):.3f}')
print(f'Runtime:  {runtime:.2f}s')
print()
print(classification_report(y_test_labels, y_pred))

report = classification_report(y_test_labels, y_pred, output_dict=True)
meta = {
    'model': 'cnn_scratch',
    'accuracy': report['accuracy'],
    'macro_f1': report['macro avg']['f1-score'],
    'negative_f1': report['negative']['f1-score'],
    'neutral_f1': report['neutral']['f1-score'],
    'positive_f1': report['positive']['f1-score'],
    'runtime_seconds': runtime
}
best_model.save('../models/images/fits/cnn_scratch.keras')
with open('../models/images/json/cnn_scratch_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved.')